In [1]:
from backend.backend import * 
import argparse
import os
from rich.console import Console
from tnn_mdls.func_mdls import *
from tnn_mdls.tb_func_mdls import *
from tnn_mdls.sim_utils import *
import time
from veriloggen import *
import copy
from model import Model
import numpy as np

In [2]:
# def add_to_dump_from_file(dump, dut, filename):
#     ports = dut.all_ports
#     f = open(filename)
#     while True:
#         line = f.readline()
#         if not line:
#             dump.add(simulation.finish())
#             break
#         else:
#             line = line.rstrip()
#             if line:
#                 dump.add(eval(line.rstrip()))

In [2]:
# Add manually configured testbenches to dump
tnn_func = TNN_Functions('0', tnn7_en=0)

m = Module('test_neuron_body')

# parameters
tres = 3
wres = 4
ip_size = m.Parameter('IP_SIZE', 4)
thres = m.Parameter('THRESHOLD', 13)
wres_v = m.Parameter('WRES', wres)

nb, bdy_clk = tnn_func.Neuronbody(ip_size=ip_size.value, thres=thres.value, wres=wres_v.value)
dut = Submodule(m, nb, 'dut')

#acc_in = dut['acc_in']

# Setup waveform dump and simulation environment
dump, clk, grst, rstb = init_dump(dut, m)

grst_gen(m, ((2**tres)+(2**wres)))

add_to_dump_from_file(dump, dut, "tnn_mdls/neuronbody_test")

In [3]:
gen_file, rtl_path = gen_verilog(module = m, filename = 'tb_test.sv')

In [6]:
# Trying to generate inputs randomly

In [2]:
import random

In [79]:
tres = 3
wres = 4

In [128]:
weight_in = 0
e_in = 0
e_out = 0
delay = 0
e_ins = [0]
e_outs = [0]
delays = []
input_prob = 0.75
output_prob = 0.75
if_input_spike = np.random.choice([0,1], p=[1-input_prob, input_prob])
if_output_spike = np.random.choice([0,1], p=[1-output_prob, output_prob])

# Generate spike times
if if_input_spike:
    e_in_time = random.randint(1, (2**tres)-1)
    e_in_reset = e_in_time + (2**wres) - (2**tres)
else:
    e_in_time = -1
    e_in_reset = -1
if if_output_spike:
    e_out_time = random.randint(1, (2**tres))
    e_out_reset = e_out_time + (2**wres) - (2**tres)
else:
    e_out_time = -1
    e_out_reset = -1

print(e_in_time, e_out_time)

# t-window
for t in range(2**tres):
    if e_in_time == t:
        e_in = 1
        e_ins += [1]
        e_outs += [e_out]
        delays += [delay]
        delay = 0
    if e_out_time == t:
        e_out = 1
        e_outs += [1]
        e_ins += [e_in]
        delays += [delay]
        delay = 0
    delay += 1

# w-window
e_ins += [e_in]
e_outs += [e_out]
delays += [delay]

for t in range(2**wres):         
    if e_in_reset == t:
        e_in = 0
        e_ins += [0]
        e_outs += [e_out]
        delays += [delay]
        delay = 0
    if e_out_reset == t:
        e_out = 0
        e_outs += [0]
        e_ins += [e_in]
        delays += [delay]
        delay = 0
    
    delay += 1
delays += [delay]

-1 1


In [129]:
e_in_time, e_out_time

(-1, 1)

In [130]:
e_in_reset, e_out_reset

(-1, 9)

In [131]:
delays

[1, 7, 16, 7]

In [132]:
e_ins, e_outs

([0, 0, 0, 0], [0, 1, 1, 0])

In [206]:
def neuronbody_test(ip_size=4, thres=13, tres=4, wres=4, wave=10, verbose=False):
    f = open("neuronbody_test_random", "w")
    for w in range(wave):
        f.write('# Wave: '+ str(w) +'\n')
        
        # Generate spike times
        spike_times = []
        for i in range(ip_size):
            spike_times += [random.randint(1, (2**tres)-1)]
        spike_times = np.array(spike_times)
        reset_times = spike_times+(2**wres-1)-(2**tres-1)
        
        if verbose:
            print('spike_times: ', spike_times)
        
        # tres window
        if verbose:
            print('t-window')
            
        in_bits = ['0'] * ip_size
        delays = []
        acc_ins = [0]
        delay = 0
        for t in range(2**tres):
            if t in spike_times:
                delays += [delay]
                delay = 0
                time_matches = np.where(spike_times == t)[0]

                # Flip bits of time spikes
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '1'

                # Calculate acc_in
                acc_string = ''
                for i in range(ip_size):
                    acc_string += in_bits[i]
                acc_ins += [int(acc_string, 2)]
            
            if verbose:
                print(in_bits)

            delay+=1

        acc_ins += [int(acc_string, 2)]
        delays+=[delay]
        
        # wres window
        if verbose:
            print('w-window')
        delay = 0
        for t in range(2**wres):
            if t in reset_times:
                delays += [delay]
                delay = 0
                time_matches = np.where(reset_times == t)[0]
                for i in range(len(time_matches)):
                    index = time_matches[i]
                    in_bits[index] = '0'

                # Calculate acc_in
                acc_string = ''
                for i in range(ip_size):
                    acc_string += in_bits[i]
                acc_ins += [int(acc_string, 2)]
                
            if verbose:
                print(in_bits)

            delay+=1
        delays+=[delay]
        
        if verbose:
            print(acc_ins, delays)
        
        # Write loop
        for i in range(len(acc_ins)):
            f.write('acc_in('+str(acc_ins[i])+'),\n')
            f.write('Delay('+str(delays[i])+'),\n')
            f.write('\n')
    f.close()
    
def stdp_test(tres=3, wres=4, input_prob=0.75, output_prob=0.75, wave=10, verbose=False):
    f = open("stdp_test_random", "w")
    for w in range(wave):
        weight_in = random.randint(0, (2**wres-1))
        f.write('weight_in('+str(weight_in)+'),\n')
        f.write('# Wave: '+ str(w) +'\n')
        e_in = 0
        e_out = 0
        delay = 0
        e_ins = [0]
        e_outs = [0]
        delays = []
        if_input_spike = np.random.choice([0,1], p=[1-input_prob, input_prob])
        if_output_spike = np.random.choice([0,1], p=[1-output_prob, output_prob])

        # Generate spike times
        if if_input_spike:
            e_in_time = random.randint(1, (2**tres)-1)
        else:
            e_in_time = -1
            e_in_reset = -1
        if if_output_spike:
            e_out_time = random.randint(1, (2**tres))
        else:
            e_out_time = -1
            e_out_reset = -1

        print(e_in_time, e_out_time)

        # t-window
        for t in range(2**tres):
            if e_in_time == t:
                e_in = 1
                e_ins += [1]
                e_outs += [e_out]
                delays += [delay]
                delay = 0
            if e_out_time == t:
                e_out = 1
                e_outs += [1]
                e_ins += [e_in]
                delays += [delay]
                delay = 0
            delay += 1

        # w-window
        e_ins += [e_in]
        e_outs += [e_out]
        delays += [delay+2**wres]
        
        if verbose:
            print('delays:', delays)
            print('ein:', e_ins)
            print('eout:', e_outs)
            print('weight_in: ', weight_in)
        
        # Write loop
        for i in range(len(delays)):
            f.write('ein('+str(e_ins[i])+'),\n')
            f.write('eout('+str(e_outs[i])+'),\n')
            f.write('Delay('+str(delays[i])+'),\n')
            f.write('\n')

In [207]:
# def neuronbody_test(ip_size=4, thres=13, wres=4, wave=10):
#     f = open("neuronbody_test_random", "w")
#     pot = 0
#     for w in range(wave):
#         acc_in = random.randint(0,(2**ip_size)-1)
#         #delay = random.randint(1,(2**wres)-1)
#         delay = random.randint(1,4)
#         f.write('# Wave: '+ str(w) +'\n')
#         f.write('acc_in('+str(acc_in)+'),\n')
#         f.write('Delay('+str(delay)+'),\n')
#         f.write('\n')
#     f.close()

In [208]:
#random.seed(14)
#neuronbody_test(ip_size=4, thres=13, tres=3, wres=4, wave=5, verbose=True)
stdp_test(tres=3, wres=4, input_prob=0.75, output_prob=0.75, wave=10, verbose=True)

7 3
delays: [3, 4, 17]
ein: [0, 0, 1, 1]
eout: [0, 1, 1, 1]
weight_in:  1
1 -1
delays: [1, 23]
ein: [0, 1, 1]
eout: [0, 0, 0]
weight_in:  2
1 -1
delays: [1, 23]
ein: [0, 1, 1]
eout: [0, 0, 0]
weight_in:  9
-1 -1
delays: [24]
ein: [0, 0]
eout: [0, 0]
weight_in:  4
5 3
delays: [3, 2, 19]
ein: [0, 0, 1, 1]
eout: [0, 1, 1, 1]
weight_in:  12
-1 8
delays: [24]
ein: [0, 0]
eout: [0, 0]
weight_in:  0
7 -1
delays: [7, 17]
ein: [0, 1, 1]
eout: [0, 0, 0]
weight_in:  14
6 -1
delays: [6, 18]
ein: [0, 1, 1]
eout: [0, 0, 0]
weight_in:  15
3 -1
delays: [3, 21]
ein: [0, 1, 1]
eout: [0, 0, 0]
weight_in:  0
3 4
delays: [3, 1, 20]
ein: [0, 1, 1, 1]
eout: [0, 0, 1, 1]
weight_in:  2


In [150]:
def extract_number(line):
    result = ""
    for c in line:
        if c.isnumeric():
            result+=c
    return int(result)

In [152]:
# Take (randomly generated) files and parse through them to extract inputs and delays
ip_size=4
thres=13
wres=4
wave=20
f = open("neuronbody_test_random", "r")
delays = []
input_names = ['acc_in']
num_waves = 20
inputs = np.zeros((len(input_names), num_waves))
inputs_mask = np.zeros((len(input_names), num_waves))
current_wave = 0

while True:
    line = f.readline()
    if not line:
        break
    else:
        line = line.rstrip()
        if line.startswith('Delay'):
            #print(line)
            delays += [extract_number(line)]
        elif line.startswith('# Wave'):
            #print(line)
            current_wave = extract_number(line)
        elif line and not line.startswith('#'):
            #print(line)
            input_index = input_names.index(line.split('(')[0])
            inputs[input_index][current_wave] = extract_number(line)
            inputs_mask[input_index][current_wave] = 1

In [153]:
# Apply mask to remove invalid inputs not in waves
masked_inputs = np.zeros((len(input_names), num_waves))
for i in range(inputs.shape[0]):
    for j in range(inputs.shape[1]):
        if inputs_mask[i][j] == 1:
            masked_inputs[i][j] = inputs[i][j]
        elif inputs_mask[i][j] == 0 and j!=0:
            masked_inputs[i][j] = masked_inputs[i][j-1]
masked_inputs

array([[ 9., 11.,  1.,  9., 11.,  6.,  8.,  2., 14.,  3.,  2., 13., 12.,
        12., 15.,  5.,  6.,  3., 10.,  1.]])

In [154]:
# Generate input matrix for every cycle
delays = np.array(delays)
delay_cumsum = np.cumsum(delays)
delay_cumsum = np.insert(delay_cumsum, 0, 0)

total_sim_time = sum(delays)
input_matrix = np.zeros((inputs.shape[0], total_sim_time))

for t in range(total_sim_time):
    if t in delay_cumsum:
        for i in range(inputs.shape[0]):
            input_matrix[i][t] = masked_inputs[i][np.where(delay_cumsum==t)[0][0]]
    elif t != 0:
        for i in range(inputs.shape[0]):
            input_matrix[i][t] = input_matrix[i][t-1]
input_matrix

array([[ 9.,  9.,  9.,  9., 11.,  1.,  1.,  9.,  9.,  9., 11., 11., 11.,
         6.,  6.,  6.,  8.,  8.,  8.,  2.,  2.,  2., 14., 14., 14.,  3.,
         2.,  2., 13., 13., 13., 13., 12., 12., 12., 12., 12., 12., 15.,
         5.,  5.,  5.,  5.,  6.,  6.,  6.,  6.,  3.,  3.,  3., 10.,  1.,
         1.,  1.]])

In [159]:
# Compute expected body_pot and output spikes
pot = 0
grst_period = 24
acc_in = input_matrix[0]

for t in range(total_sim_time):
    if t==grst_period:
        pot = 0
    else:
        pot += bin(int(acc_in[t])).count("1")
    print(pot)
    if pot >= thres:
        print('spike at t=', t)
        pot = 0

2
4
6
8
11
12
13
spike at t= 6
2
4
6
9
12
15
spike at t= 12
2
4
6
7
8
9
10
11
12
15
spike at t= 22
3
0
2
3
4
7
10
13
spike at t= 30
3
5
7
9
11
13
spike at t= 36
2
6
8
10
12
14
spike at t= 42
2
4
6
8
10
12
14
spike at t= 49
2
3
4
5
